In [16]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [17]:
import sys
print(sys.executable)

c:\Python313\python.exe


In [18]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chhap\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\chhap\AppData\Roaming\nltk_data...


True

In [19]:
df_raw = pd.read_csv("dataset/Resume.csv")

print("Shape of raw data:", df_raw.shape)
print("Columns:", df_raw.columns)

df_raw.head()

Shape of raw data: (2484, 4)
Columns: Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [20]:
df_clean1 = df_raw[['Resume_str', 'Category']].copy()

print("Shape after selecting useful columns:", df_clean1.shape)

df_clean1.head()

Shape after selecting useful columns: (2484, 2)


,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [21]:
df_clean1.isnull().sum()

Resume_str    0
Category      0
dtype: int64

In [23]:
df_clean2 = df_clean1.copy()

# Remove all HTML entities like &nbsp;, &amp;, etc.
df_clean2['text'] = df_clean2['Resume_str'].apply(
    lambda x: re.sub(r'&[a-zA-Z]+;', ' ', x)
)

df_clean2[['Resume_str', 'text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ...","HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...,HR DIRECTOR Summary Over 2...


In [24]:
df_clean3 = df_clean2.copy()

df_clean3['text'] = df_clean3['text'].str.lower()

df_clean3[['Resume_str','text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator/marketing associate\...
1,"HR SPECIALIST, US HR OPERATIONS ...","hr specialist, us hr operations ..."
2,HR DIRECTOR Summary Over 2...,hr director summary over 2...


In [25]:
df_clean4 = df_clean3.copy()

# Remove any HTML entity like &nbsp; &amp; etc.
df_clean4['text'] = df_clean4['text'].apply(
    lambda x: re.sub(r'&[a-zA-Z]+;', ' ', x)
)

# Remove non-breaking space unicode character if present
df_clean4['text'] = df_clean4['text'].str.replace('\xa0', ' ', regex=False)

df_clean4[['Resume_str','text']].head(3)

,Resume_str,text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator/marketing associate\...
1,"HR SPECIALIST, US HR OPERATIONS ...","hr specialist, us hr operations ..."
2,HR DIRECTOR Summary Over 2...,hr director summary over 2...


In [26]:
df_clean5 = df_clean4.copy()

# Remove emails
df_clean5['text'] = df_clean5['text'].apply(
    lambda x: re.sub(r'\S+@\S+', '', x)
)

df_clean5[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over 2...


In [27]:
df_clean6 = df_clean5.copy()

# Remove URLs
df_clean6['text'] = df_clean6['text'].apply(
    lambda x: re.sub(r'http\S+|www\S+', '', x)
)

df_clean6[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over 2...


In [28]:
df_clean7 = df_clean6.copy()

# Remove numbers
df_clean7['text'] = df_clean7['text'].apply(
    lambda x: re.sub(r'\d+', '', x)
)

df_clean7[['text']].head(3)

,text
0,hr administrator/marketing associate\...
1,"hr specialist, us hr operations ..."
2,hr director summary over ...


In [30]:
df_clean8 = df_clean7.copy()

# Replace punctuation with space instead of removing
df_clean8['text'] = df_clean8['text'].apply(
    lambda x: re.sub(r'[^\w\s]', ' ', x)
)

df_clean8[['text']].head(3)

,text
0,hr administrator marketing associate\...
1,hr specialist us hr operations ...
2,hr director summary over ...


In [31]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [32]:
df_clean9 = df_clean8.copy()

df_clean9['text'] = df_clean9['text'].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

df_clean9[['text']].head(3)

,text
0,hr administrator marketing associate hr admini...
1,hr specialist us hr operations summary versati...
2,hr director summary over years experience in r...


In [33]:
df_clean10 = df_clean9.copy()

df_clean10['text'] = df_clean10['text'].apply(
    lambda x: " ".join([
        word for word in x.split()
        if word not in stop_words and len(word) > 2
    ])
)

df_clean10[['text']].head(3)

,text
0,administrator marketing associate administrato...
1,specialist operations summary versatile media ...
2,director summary years experience recruiting p...


In [34]:
df_clean11 = df_clean10.copy()

df_clean11['text'] = df_clean11['text'].apply(
    lambda x: " ".join([
        lemmatizer.lemmatize(word)
        for word in x.split()
    ])
)

df_clean11[['text']].head(3)

,text
0,administrator marketing associate administrato...
1,specialist operation summary versatile medium ...
2,director summary year experience recruiting pl...


In [35]:
df_final = df_clean11.copy()

df_final = df_final[df_final['text'].str.strip() != ""]

print("Final shape:", df_final.shape)

Final shape: (2483, 3)


In [36]:
print(df_final.columns)

Index(['Resume_str', 'Category', 'text'], dtype='str')


In [37]:
df_final = df_clean11.copy()

df_final = df_final.rename(columns={'text': 'cleaned_resume'})

print(df_final.columns)

Index(['Resume_str', 'Category', 'cleaned_resume'], dtype='str')


In [38]:
df_final = df_final[['Category', 'cleaned_resume']]

print(df_final.shape)
df_final.head()

(2484, 2)


,Category,cleaned_resume
0,HR,administrator marketing associate administrato...
1,HR,specialist operation summary versatile medium ...
2,HR,director summary year experience recruiting pl...
3,HR,specialist summary dedicated driven dynamic ye...
4,HR,manager skill highlight skill department start...


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [40]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,1)   # unigrams only for now
)

In [41]:
X = tfidf.fit_transform(df_final['cleaned_resume'])

print("TF-IDF Matrix Shape:", X.shape)

TF-IDF Matrix Shape: (2484, 5000)


In [43]:
print(df_final.shape)
print(len(df_final['cleaned_resume']))

(2484, 2)
2484


In [44]:
y = df_final['Category']

print("Target Shape:", y.shape)

Target Shape: (2484,)


In [45]:
feature_names = tfidf.get_feature_names_out()

print("First 20 features:")
print(feature_names[:20])

First 20 features:
['aa' 'aaa' 'abc' 'ability' 'able' 'abreast' 'abroad' 'absence' 'abuse'
 'academic' 'academy' 'accelerate' 'accelerated' 'accept' 'acceptable'
 'acceptance' 'accepted' 'accepting' 'access' 'accessory']


In [ ]:
import numpy as np

zero_rows = np.sum(X.sum(axis=1) == 0)
print("Number of zero rows:", zero_rows)

In [47]:
zero_indices = np.where(X.sum(axis=1) == 0)[0]
print("Zero row index:", zero_indices)


Zero row index: [656]


In [48]:
X = np.delete(X.toarray(), zero_indices, axis=0)
y = y.drop(y.index[zero_indices]).reset_index(drop=True)

In [49]:
print("New X shape:", X.shape)
print("New y shape:", y.shape)

zero_rows = np.sum(np.sum(X, axis=1) == 0)
print("Zero rows after fix:", zero_rows)

New X shape: (2483, 5000)
New y shape: (2483,)
Zero rows after fix: 0


In [50]:
df_final.to_csv("cleaned_resume1.csv", index=False)

In [51]:
df_check = pd.read_csv("cleaned_resume1.csv")

print(df_check.shape)
df_check.head()

(2484, 2)


,Category,cleaned_resume
0,HR,administrator marketing associate administrato...
1,HR,specialist operation summary versatile medium ...
2,HR,director summary year experience recruiting pl...
3,HR,specialist summary dedicated driven dynamic ye...
4,HR,manager skill highlight skill department start...


In [55]:
df_check.isnull().sum()

Category          0
cleaned_resume    0
dtype: int64

In [53]:
df_check[df_check['cleaned_resume'].isnull()]

,Category,cleaned_resume
656,BUSINESS-DEVELOPMENT,NaN


In [54]:
df_check = df_check.dropna(subset=['cleaned_resume'])

print(df_check.shape)
df_check.isnull().sum()

(2483, 2)


Category          0
cleaned_resume    0
dtype: int64

In [56]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df_check['cleaned_resume'])
y = df_check['Category']

print(X.shape)
print(y.shape)

(2483, 5000)
(2483,)


In [57]:
df_final.to_csv("cleaned_resume1.csv", index=False)

In [60]:
import pandas as pd

df = pd.read_csv("cleaned_resume1.csv")

# Remove any null rows (safety)
df = df.dropna(subset=['cleaned_resume'])

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2483, 2)


,Category,cleaned_resume
0,HR,administrator marketing associate administrato...
1,HR,specialist operation summary versatile medium ...
2,HR,director summary year experience recruiting pl...
3,HR,specialist summary dedicated driven dynamic ye...
4,HR,manager skill highlight skill department start...


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['cleaned_resume'])
y = df['Category']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2483, 5000)
y shape: (2483,)


In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1986, 5000)
Test shape: (497, 5000)


In [63]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [64]:
y_pred_nb = model_nb.predict(X_test)

In [65]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_nb = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", accuracy_nb)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.5432595573440644

Classification Report:

                        precision    recall  f1-score   support

            ACCOUNTANT       0.50      0.88      0.64        24
              ADVOCATE       0.36      0.42      0.38        24
           AGRICULTURE       1.00      0.08      0.14        13
               APPAREL       0.67      0.11      0.18        19
                  ARTS       0.75      0.14      0.24        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.82      0.58      0.68        24
               BANKING       0.93      0.61      0.74        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.34      0.83      0.48        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.65      0.68      0.67        22
            CONSULTANT       1.00      0.04      0.08        23
              DESIGNER       0.80    

C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [67]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

In [68]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy_lr)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.6519114688128773

Classification Report:

                        precision    recall  f1-score   support

            ACCOUNTANT       0.67      0.83      0.74        24
              ADVOCATE       0.35      0.50      0.41        24
           AGRICULTURE       1.00      0.46      0.63        13
               APPAREL       0.60      0.16      0.25        19
                  ARTS       0.42      0.24      0.30        21
            AUTOMOBILE       0.00      0.00      0.00         7
              AVIATION       0.86      0.75      0.80        24
               BANKING       0.84      0.70      0.76        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.47      0.79      0.59        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.85      0.77      0.81        22
            CONSULTANT       0.44      0.17      0.25        23
              DESIGNER       

C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [69]:
model_svm = LinearSVC()
model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)

In [70]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", accuracy_svm)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

Linear SVM Accuracy: 0.704225352112676

Classification Report:

                        precision    recall  f1-score   support

            ACCOUNTANT       0.70      0.79      0.75        24
              ADVOCATE       0.67      0.75      0.71        24
           AGRICULTURE       0.88      0.54      0.67        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.50      0.38      0.43        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.82      0.75      0.78        24
               BANKING       0.86      0.78      0.82        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.56      0.83      0.67        24
                  CHEF       0.86      0.75      0.80        24
          CONSTRUCTION       0.86      0.82      0.84        22
            CONSULTANT       0.80      0.35      0.48        23
              DESIGNER       0.82      

In [71]:
print(classification_report(y_test, y_pred_svm))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.70      0.79      0.75        24
              ADVOCATE       0.67      0.75      0.71        24
           AGRICULTURE       0.88      0.54      0.67        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.50      0.38      0.43        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.82      0.75      0.78        24
               BANKING       0.86      0.78      0.82        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.56      0.83      0.67        24
                  CHEF       0.86      0.75      0.80        24
          CONSTRUCTION       0.86      0.82      0.84        22
            CONSULTANT       0.80      0.35      0.48        23
              DESIGNER       0.82      0.86      0.84        21
         DIGITAL-MEDIA       0.73      

In [73]:
from xgboost import XGBClassifier

In [74]:
X_dense = X.toarray()

In [75]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

In [82]:
from sklearn.model_selection import train_test_split

X_train_dense, X_test_dense, y_train_enc, y_test_enc = train_test_split(
    X_dense,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [83]:
model_xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=len(y.unique()),
    eval_metric='mlogloss',
    use_label_encoder=False
)

In [79]:
from sklearn.preprocessing import LabelEncoder

In [80]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Encoded labels example:", y_encoded[:10])

Encoded labels example: [19 19 19 19 19 19 19 19 19 19]


In [84]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False
)

model_xgb.fit(X_train_dense, y_train_enc)

C:\Users\chhap\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [10:44:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [85]:
y_pred_xgb = model_xgb.predict(X_test_dense)

In [86]:
from sklearn.metrics import accuracy_score

accuracy_xgb = accuracy_score(y_test_enc, y_pred_xgb)
print("XGBoost Accuracy:", accuracy_xgb)

XGBoost Accuracy: 0.7303822937625755
